# Proyecto 03: Modificando el Marco de Referencia

## 0. Código y Funciones Auxiliares

In [197]:
# Import de librerías para la realización del proyecto

from pathlib import Path
from functools import reduce

import cv2
import numpy as np
import matplotlib.pyplot as plt

### 0.1 Carga de Imágenes

In [3]:
Path("./results").mkdir(exist_ok=True)

### 0.2 Transformaciones Afines

In [196]:
def translation_matrix(
        x_move: float = 0,
        y_move: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de traslación. Toma 
    de entrada el movimiento 
    horizontal y vertical.
    """

    matrix = np.array([
        [1, 0, x_move],
        [0, 1, y_move],
        [0, 0, 1]
    ], dtype=float)
    return matrix

In [50]:
def rotation_matrix(angle: float) -> np.ndarray:
    """
    Función para generar la 
    matriz de rotación. Toma 
    de entrada el ángulo de 
    rotación.
    """

    cos_angle = np.cos(angle)
    sin_angle = np.sin(angle)
    matrix = np.array([
        [cos_angle, sin_angle, 0],
        [-sin_angle, cos_angle, 0],
        [0, 0, 1]
    ], dtype=float)
    return matrix

In [ ]:
def centered_rotation_matrix(
        center: np.ndarray = np.array([0, 0, 1]),
        angle: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la 
    matriz de rotación centrada 
    respecto al centro de la 
    imagen. Toma de entrada el 
    centro de la imagen y el 
    ángulo de rotation.
    """

    rotation = rotation_matrix(angle)
    transformed_center = rotation@center
    adjust = (center-transformed_center)
    centered_rotation = translate_matrix(adjust[0], adjust[1])@rotation
    return centered_rotation

In [ ]:
def reflection_matrix(
        reflect_x: bool = False,
        reflect_y: bool = False,
    ) -> np.ndarray:
    """
    Función para generar la matriz 
    de reflexión sobre los ejes. Toma 
    de entrada si se aplica la reflexión 
    individualmente sobre los ejes x y y.
    """

    value_x = -1 if reflect_x else 1
    value_y = -1 if reflect_y else 1
    reflection = np.array([
        [value_y, 0, 0],
        [0, value_x, 0],
        [0, 0, 1],
    ])
    return reflection

In [ ]:
def scaling_matrix(
        scale_x: float = 1,
        scale_y: float = 1,
    ) -> np.ndarray:
    """
    Función para generar la matriz 
    de escala sobre los ejes. 
    Toma de entrada las nuevas escalas 
    de los ejes x y y.
    """

    scaling = np.array([
        [scale_x, 0, 0],
        [0, scale_y, 0],
        [0, 0, 1],
    ])
    return scaling

In [ ]:
def shearing_matrix(
        shift_x: float = 0,
        shift_y: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la matriz 
    de cizallamiento sobre los ejes. 
    Toma de entrada los desplazamientos 
    de los ejes x y y.
    """

    shearing = np.array([
        [1, shift_x, 0],
        [shift_y, 1, 0],
        [0, 0, 1]
    ])
    return shearing

In [209]:
def affine_transformation_matrix(
        x_move: float = 0,
        y_move: float = 0,
        center: np.ndarray = np.array([0, 0, 1]),
        angle: float = 0,
        reflect_x: bool = False,
        reflect_y: bool = False,
        scale_x: float = 1, 
        scale_y: float = 1,
        shift_x: float = 0,
        shift_y: float = 0,
    ) -> np.ndarray:
    """
    Función para generar la matriz general 
    de una transformación afín. Toma de 
    parámetros los definidos para cada 
    matriz de transformación individualmente.
    """

    transformations = [
        translation_matrix(x_move, y_move),
        centered_rotation_matrix(center, angle),
        reflection_matrix(reflect_x, reflect_y),
        scaling_matrix(scale_x, scale_y),
        shearing_matrix(shift_x, shift_y),
    ]
    return reduce(lambda t_1, t_2 : t_1@t_2, transformations)